# LLM Zoomcamp 2026 — Homework 2: Vector Search

Rebuilt notebook with working code for all 6 questions.

In [1]:
from gitsource import GithubRepositoryDataReader
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [2]:
len(documents)

72

## Q1: Embedding dimension and first vector value

**Answer: dimension = 384, v[0] ≈ -0.02058203437252893**

In [4]:
from embedder import Embedder
embedder = Embedder()

In [5]:
query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)
print(len(v))
print(v[0])

384
-0.02058203437252893


## Q2: Cosine similarity

**Answer: cosine_sim ≈ 0.36107027225589694**

In [6]:
target_filename = "02-vector-search/lessons/07-sqlitesearch-vector.md"
doc = next(d for d in documents if d["filename"] == target_filename)
doc_vector = embedder.encode(doc["content"])
cosine_sim = v.dot(doc_vector)
print(cosine_sim)

0.36107027225589694


## Q3: Chunking and best matching chunk

**Answer: 02-vector-search/lessons/07-sqlitesearch-vector.md**

In [9]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
import numpy as np
X = np.array([embedder.encode(c["content"]) for c in chunks])
scores = X.dot(v)
best_idx = scores.argmax()
print(chunks[best_idx]["filename"])

02-vector-search/lessons/07-sqlitesearch-vector.md


## Q4: Vector search with minsearch

**Answer: 04-evaluation/lessons/05-search-metrics.md**

In [10]:
from minsearch import VectorSearch
vindex = VectorSearch()
vindex.fit(X, chunks)
q4_query = "What metric do we use to evaluate a search engine?"
q4_vector = embedder.encode(q4_query)
results = vindex.search(q4_vector, num_results=5)
print(results[0]["filename"])

04-evaluation/lessons/05-search-metrics.md


## Q5: Text search vs vector search comparison

**Answer: 02-vector-search/lessons/08-pgvector.md** (only appears in vector results)

In [11]:
from minsearch import Index
text_index = Index(text_fields=["content"])
text_index.fit(chunks)
q5_query = "How do I store vectors in PostgreSQL?"
q5_vector = embedder.encode(q5_query)
text_results = text_index.search(q5_query, num_results=5)
vector_results = vindex.search(q5_vector, num_results=5)
text_files = {r["filename"] for r in text_results}
vector_files = {r["filename"] for r in vector_results}
only_in_vector = vector_files - text_files
print(only_in_vector)

{'02-vector-search/lessons/08-pgvector.md'}


## Q6: Reciprocal Rank Fusion (RRF) — hybrid search

**Answer: 01-agentic-rag/lessons/13-function-calling.md**

Formula: RRF(d) = sum over lists of 1 / (k + rank(d))

In [13]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [14]:
q6_query = "How do I give the model access to tools?"
q6_vector = embedder.encode(q6_query)
vector_results = vindex.search(q6_vector, num_results=5)
text_results = text_index.search(q6_query, num_results=5)
results = rrf([vector_results, text_results])
print(results[0]["filename"])

01-agentic-rag/lessons/13-function-calling.md


## Summary of Answers

| Question | Answer |
|---|---|
| Q1 — Embedding dimension | 384 |
| Q1 — First vector value | ≈ -0.02 |
| Q2 — Cosine similarity | ≈ 0.36 |
| Q3 — Best matching chunk | 02-vector-search/lessons/07-sqlitesearch-vector.md |
| Q4 — Top vector search result | 04-evaluation/lessons/05-search-metrics.md |
| Q5 — Doc only in vector results | 02-vector-search/lessons/08-pgvector.md |
| Q6 — Top RRF hybrid result | 01-agentic-rag/lessons/13-function-calling.md |